In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from datetime import datetime
import logging

## Declare Variable
AUDIT_LOG_TABLE = "AuditLog"
DOCUMENT_REGISTRY_TABLE = "documentregistry"

messageBody = "{\"topic\":\"/subscriptions/80f18dcf-f863-416f-99e4-ec6dabaa511d/resourceGroups/int-fabric-rg/providers/Microsoft.Storage/storageAccounts/intfabriccustdata\",\"subject\":\"/blobServices/default/containers/storage-cust-data-container/blobs/UPMC/MA/2026/2026-08-13/IMG_20260803_173155 (1).jpg\",\"eventType\":\"Microsoft.Storage.BlobCreated\",\"id\":\"80036003-c01e-003e-43b0-2bbeb506c46c\",\"data\":{\"api\":\"PutBlob\",\"requestId\":\"80036003-c01e-003e-43b0-2bbeb5000000\",\"eTag\":\"0x8DEF9C7B3C18025\",\"contentType\":\"image/jpeg\",\"contentLength\":4687339,\"blobType\":\"BlockBlob\",\"accessTier\":\"Default\",\"url\":\" https://intfabriccustdata.blob.core.windows.net/storage-cust-data-container/UPMC/MA/2026/2026-08-13/IMG_20260803_173155  (1).jpg\",\"sequencer\":\"000000000000000000000000000268750000000000061fea\",\"storageDiagnostics\":{\"batchId\":\"19d53f63-1006-002d-00b0-2b9ab9000000\"}},\"dataVersion\":\"\",\"metadataVersion\":\"1\",\"eventTime\":\"2026-08-14T05:48:44.2810141Z\"}"


StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 12, Finished, Available, Finished, False)

In [1]:
# messageBody is likely this JSON as a string — parse it
import json
import re
from datetime import datetime

event_data = json.loads(messageBody) if isinstance(messageBody, str) else messageBody

# ---- Top-level event fields ----
event_type = event_data.get("eventType")           
event_id   = event_data.get("id")                   
event_time = event_data.get("eventTime")
subject    = event_data.get("subject", "")
topic      = event_data.get("topic", "")  

# ---- Nested "data" fields ----
data           = event_data.get("data", {})
blob_url       = data.get("url")
content_type   = data.get("contentType")
content_length = data.get("contentLength")
blob_type      = data.get("blobType")
etag           = data.get("eTag")
api            = data.get("api")

# ---- Derived values ----
# Full path inside the container, e.g. "UPMC/MA/2026/2026-08-13/CareFirst_....pdf"
blob_relative_match = re.search(r"/blobs/(.+)$", subject)
blob_relative_path = blob_relative_match.group(1) if blob_relative_match else None

# Just the filename (last segment)
file_name = blob_relative_path.split("/")[-1] if blob_relative_path else (blob_url.split("/")[-1] if blob_url else None)

container_match = re.search(r"/containers/([^/]+)/", subject)
container_name = container_match.group(1) if container_match else None

blob_path = blob_url  # full blob URL as BlobPath

received_time = event_time

eventId = event_id
eventType = event_type


StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 13, Finished, Available, Finished, False)

In [ ]:
logger = logging.getLogger("Validation")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False

StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 14, Finished, Available, Finished, False)

In [ ]:
def write_audit_log(
    process_id,
    status,
    stage
):
    try:
        schema = StructType([
            StructField("ProcessId", StringType(), False),
            StructField("Stage", StringType(), False),
            StructField("Status", StringType(), False),
            StructField("ActionTime", TimestampType(), False)
        ])

        audit_data = [(
            str(process_id),
            str(stage),
            str(status),
            datetime.utcnow()
        )]

        audit_df = spark.createDataFrame(audit_data, schema=schema)

        audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(AUDIT_LOG_TABLE)

        logger.info(
            f"AUDIT LOG WRITTEN | "
            f"ProcessId={process_id} | "
            f"Stage={stage} | "
            f"Status={status}")

    except Exception as e:
        logger.error(
            f"AUDIT LOG WRITE FAILED | "
            f"ProcessId={process_id} | "
            f"Stage={stage} | "
            f"Status={status} | "
            f"Error={str(e)}")
        raise

process_id = eventId if eventId else "12345"

print(process_id)
logger.info(
    f"PARAMETERS RECEIVED | "
    f"eventId={eventId} | "
    f"eventType={eventType} | "
    f"messageBody={messageBody}"
)

# ---- Example call ----
write_audit_log(
    process_id=process_id,
    stage="INGESTION",
    status="COMPLETED"
)


StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 15, Finished, Available, Finished, False)

2026-08-14 07:14:09,566 | INFO | PARAMETERS RECEIVED | eventId=80036003-c01e-003e-43b0-2bbeb506c46c | eventType=Microsoft.Storage.BlobCreated | messageBody={"topic":"/subscriptions/80f18dcf-f863-416f-99e4-ec6dabaa511d/resourceGroups/int-fabric-rg/providers/Microsoft.Storage/storageAccounts/intfabriccustdata","subject":"/blobServices/default/containers/storage-cust-data-container/blobs/UPMC/MA/2026/2026-08-13/IMG_20260803_173155 (1).jpg","eventType":"Microsoft.Storage.BlobCreated","id":"80036003-c01e-003e-43b0-2bbeb506c46c","data":{"api":"PutBlob","requestId":"80036003-c01e-003e-43b0-2bbeb5000000","eTag":"0x8DEF9C7B3C18025","contentType":"image/jpeg","contentLength":4687339,"blobType":"BlockBlob","accessTier":"Default","url":" https://intfabriccustdata.blob.core.windows.net/storage-cust-data-container/UPMC/MA/2026/2026-08-13/IMG_20260803_173155  (1).jpg","sequencer":"000000000000000000000000000268750000000000061fea","storageDiagnostics":{"batchId":"19d53f63-1006-002d-00b0-2b9ab9000000"}

80036003-c01e-003e-43b0-2bbeb506c46c


2026-08-14 07:14:14,920 | INFO | AUDIT LOG WRITTEN | ProcessId=80036003-c01e-003e-43b0-2bbeb506c46c | Stage=INGESTION | Status=COMPLETED


In [ ]:
def write_document_registry(
    process_id,
    client_id,
    file_name,
    blob_path,
    lakehouse_path,
    file_hash
):
    try:
        schema = StructType([
            StructField("ProcessId", StringType(), False),
            StructField("ClientId", StringType(), False),
            StructField("FileName", StringType(), False),
            StructField("BlobPath", StringType(), False),
            StructField("LakehousePath", StringType(), False),
            StructField("ReceivedTime", TimestampType(), False),
            StructField("FileHash", StringType(), False)
            ])

        registry_data = [(
            str(process_id),
            str(client_id),
            str(file_name),
            str(blob_path),
            str(lakehouse_path),
            datetime.utcnow(),
            str(file_hash)
        )]

        registry_df = spark.createDataFrame(registry_data, schema=schema)

        registry_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(DOCUMENT_REGISTRY_TABLE)

        logger.info(
            f"DOCUMENT REGISTRY WRITTEN | "
            f"ProcessId={process_id} | "
            f"ClientId={client_id} | "
            f"FileName={file_name} | "
            f"BlobPath={blob_path} | "
            f"LakehousePath={lakehouse_path} | "
            f"FileHash={file_hash}")

    except Exception as e:
        logger.error(
            f"DOCUMENT REGISTRY WRITE FAILED | "
            f"ProcessId={process_id} | "
            f"ClientId={client_id} | "
            f"FileName={file_name} | "
            f"Error={str(e)}")
        raise


process_id = eventId if eventId else "12345"

print(process_id)
logger.info(
    f"PARAMETERS RECEIVED | "
    f"eventId={eventId} | "
    f"eventType={eventType} | "
    f"messageBody={messageBody}"
)

# ---- Example call ----
write_document_registry(
    process_id=process_id,
    client_id=process_id,
    file_name=file_name,
    blob_path=blob_url,
    lakehouse_path=blob_url,
    file_hash=file_name
)

StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 16, Finished, Available, Finished, False)

80036003-c01e-003e-43b0-2bbeb506c46c


2026-08-14 07:14:19,098 | INFO | DOCUMENT REGISTRY WRITTEN | ProcessId=80036003-c01e-003e-43b0-2bbeb506c46c | ClientId=80036003-c01e-003e-43b0-2bbeb506c46c | FileName=IMG_20260803_173155 (1).jpg | BlobPath= https://intfabriccustdata.blob.core.windows.net/storage-cust-data-container/UPMC/MA/2026/2026-08-13/IMG_20260803_173155  (1).jpg | LakehousePath= https://intfabriccustdata.blob.core.windows.net/storage-cust-data-container/UPMC/MA/2026/2026-08-13/IMG_20260803_173155  (1).jpg | FileHash=IMG_20260803_173155 (1).jpg


In [ ]:
import json

output_payload = {
    "processId": process_id,
    "containerName": container_name,
    "blobRelativePath": blob_relative_path
}

output_json = json.dumps(output_payload)

logger.info(f"NOTEBOOK EXIT PAYLOAD | {output_json}")

notebookutils.notebook.exit(output_json)


StatementMeta(, 5e56bf46-ed80-4bd2-b018-9356aad21ca6, 17, Finished, Available, Finished, False)

2026-08-14 07:14:20,547 | INFO | NOTEBOOK EXIT PAYLOAD | {"processId": "80036003-c01e-003e-43b0-2bbeb506c46c", "containerName": "storage-cust-data-container", "blobRelativePath": "UPMC/MA/2026/2026-08-13/IMG_20260803_173155 (1).jpg"}


ExitValue: {"processId": "80036003-c01e-003e-43b0-2bbeb506c46c", "containerName": "storage-cust-data-container", "blobRelativePath": "UPMC/MA/2026/2026-08-13/IMG_20260803_173155 (1).jpg"}